# NYC Taxi Data Pipeline: SQL Analytics
## Teresa Ferrill
### October 8, 2026

This notebook queries the processed January 2025 Parquet dataset using DuckDB. SQL summaries of daily, hourly, and pickup borough activity are compared with the pandas summaries exported by Notebook 03.  

Counts include flagged records and describe recorded activity.  Run Notebooks 02 and 03 first. Run this notebook from the notebooks folder.  

In [1]:
%pip install duckdb

   ---------------------------------------- 0.0/13.2 MB ? eta -:--:--
   -- ------------------------------------- 0.8/13.2 MB 10.3 MB/s eta 0:00:02
   -------------- ------------------------- 4.7/13.2 MB 16.6 MB/s eta 0:00:01
   ------------------------------- -------- 10.2/13.2 MB 20.8 MB/s eta 0:00:01
   ---------------------------------------- 13.2/13.2 MB 20.7 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import duckdb
import pandas as pd
from pandas.testing import assert_frame_equal

processed_path = Path(
    "../data/processed/yellow_trips_2025-01_processed.parquet"
)
report_dir = Path("../reports")

required_files = [
    processed_path,
    report_dir / "daily_trip_counts_2025-01.csv",
    report_dir / "hourly_trip_counts_2025-01.csv",
    report_dir / "pickup_borough_counts_2025-01.csv"
]

for path in required_files:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing {path}. Run Notebooks 02 and 03 first."
        )

con = duckdb.connect()

# Register a view over the Parquet file without copying it into a table.
con.read_parquet(str(processed_path)).create_view("january_trips")

total_records = con.execute(
    "SELECT COUNT(*) FROM january_trips"
).fetchone()[0]

print(f"Records available to SQL: {total_records:,}")

Records available to SQL: 3,475,204


### 1. Generate SQL Activity Summaries

Queries count records by pickup date, pickup hour, and pickup borough.  

Calendar dates and hours are included even when their counts are zero.  

Missing borough descriptions use the same label as Notebook 03.  

Each summary must reconcile to the source record count.

In [3]:
sql_daily = con.execute("""
    WITH counts AS (
        SELECT
            CAST(tpep_pickup_datetime AS DATE) AS pickup_date,
            COUNT(*) AS record_count
        FROM january_trips
        GROUP BY 1
    ),
    calendar AS (
        SELECT CAST(day AS DATE) AS pickup_date
        FROM generate_series(
            TIMESTAMP '2025-01-01',
            TIMESTAMP '2025-01-31',
            INTERVAL '1 day'
        ) AS dates(day)
    )
    SELECT
        calendar.pickup_date,
        COALESCE(counts.record_count, 0) AS record_count
    FROM calendar
    LEFT JOIN counts USING (pickup_date)
    ORDER BY calendar.pickup_date
""").df()

sql_hourly = con.execute("""
    WITH counts AS (
        SELECT
            CAST(EXTRACT(HOUR FROM tpep_pickup_datetime) AS INTEGER)
                AS pickup_hour,
            COUNT(*) AS record_count
        FROM january_trips
        GROUP BY 1
    ),
    hours AS (
        SELECT CAST(hour AS INTEGER) AS pickup_hour
        FROM range(24) AS series(hour)
    )
    SELECT
        hours.pickup_hour,
        COALESCE(counts.record_count, 0) AS record_count
    FROM hours
    LEFT JOIN counts USING (pickup_hour)
    ORDER BY hours.pickup_hour
""").df()

sql_borough = con.execute("""
    SELECT
        COALESCE(pickup_borough, 'Missing borough description')
            AS pickup_borough,
        COUNT(*) AS record_count
    FROM january_trips
    GROUP BY 1
    ORDER BY record_count DESC, pickup_borough
""").df()

sql_borough["percent_of_records"] = (
    sql_borough["record_count"] / total_records * 100
).round(2)

for summary in [sql_daily, sql_hourly, sql_borough]:
    assert summary["record_count"].sum() == total_records

display(sql_daily.head())
display(sql_hourly)
display(sql_borough)

,pickup_date,record_count
0,2025-01-01,90188
1,2025-01-02,84832
2,2025-01-03,91250
3,2025-01-04,97804
4,2025-01-05,79624


,pickup_hour,record_count
0,0,93416
1,1,64484
2,2,43929
3,3,28492
4,4,20033
5,5,22551
6,6,50026
7,7,102581
8,8,141305
9,9,142877


,pickup_borough,record_count,percent_of_records
0,Manhattan,3089258,88.89
1,Queens,294981,8.49
2,Brooklyn,66070,1.90
3,Bronx,14741,0.42
4,Unknown,8141,0.23
5,Missing borough description,1380,0.04
6,EWR,377,0.01
7,Staten Island,256,0.01


### 2. Compare SQL and pandas Results

Compare each SQL summary with its corresponding Notebook 03 CSV.
Dates and row ordering are standardized before comparison.
Storage data-type differences are allowed; values must match exactly.

In [4]:
pandas_daily = pd.read_csv(
    report_dir / "daily_trip_counts_2025-01.csv",
    parse_dates=["pickup_date"]
)
pandas_hourly = pd.read_csv(
    report_dir / "hourly_trip_counts_2025-01.csv"
)
pandas_borough = pd.read_csv(
    report_dir / "pickup_borough_counts_2025-01.csv"
)

comparisons = [
    ("daily", sql_daily, pandas_daily, "pickup_date"),
    ("hourly", sql_hourly, pandas_hourly, "pickup_hour"),
    ("borough", sql_borough, pandas_borough, "pickup_borough")
]

validation_rows = []

for name, sql_result, pandas_result, key in comparisons:
    actual = sql_result.copy()
    expected = pandas_result.copy()

    if key == "pickup_date":
        actual[key] = pd.to_datetime(actual[key]).dt.normalize()
        expected[key] = pd.to_datetime(expected[key]).dt.normalize()

    actual = actual.sort_values(key).reset_index(drop=True)
    expected = expected.sort_values(key).reset_index(drop=True)

    assert_frame_equal(
        actual,
        expected,
        check_dtype=False,
        check_exact=True
    )

    validation_rows.append({
        "summary": name,
        "matches_pandas": True,
        "summary_rows": len(actual),
        "total_records": int(actual["record_count"].sum())
    })

    actual.to_csv(
        report_dir / f"sql_{name}_counts_2025-01.csv",
        index=False
    )

validation_report = pd.DataFrame(validation_rows)
validation_report.to_csv(
    report_dir / "sql_validation_2025-01.csv",
    index=False
)

display(validation_report)
print("All SQL summaries match the pandas summaries.")

con.close()

,summary,matches_pandas,summary_rows,total_records
0,daily,True,31,3475204
1,hourly,True,24,3475204
2,borough,True,8,3475204


All SQL summaries match the pandas summaries.


### Validation Interpretation

Successful comparisons show that DuckDB SQL and pandas produce
identical activity summaries from the processed January dataset.

This verifies agreement between the two implementations. It does
not independently establish the accuracy of the original trip records.